# Predicting Headline Success with Embedding-based Techniques

## Load dataset

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from scipy.special import logit, expit
from sklearn.linear_model import LinearRegression
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from itertools import combinations

In [2]:
import pandas as pd

train_df = pd.read_csv("dataset/processed/confirmatory-packages-topic-modeling.csv")
validation_df = pd.read_csv("dataset/processed/exploratory-packages-topic-modeling.csv")
test_df = pd.read_csv("dataset/processed/holdout-packages-topic-modeling.csv")

## Generate embeddings

In this case we will use the all-MiniLM-L6-v2 model to generate embeddings for the headlines.

In [3]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")

def generate_embeddings_dataset(df):
    df = df[["headline", "impressions", "clicks", "clickability_test_id"]].copy()
    embeddings = encoder.encode(df["headline"].tolist(), convert_to_numpy=True)
    df["embedding"] = [emb.astype(np.float32) for emb in embeddings]
    return df

In [4]:
train_embeddings_df = generate_embeddings_dataset(train_df)
validation_embeddings_df = generate_embeddings_dataset(validation_df)
test_embeddings_df = generate_embeddings_dataset(test_df)

## Generate Pairwise Dataset

Before being able to compare the results, we need to define how we will measure the accuracy of the model.

In this case, we decided to create a set of pairwise comparisons between the headlines in the same Test. Our model will learn to predict which headline is more clickable.

The accuracy will be measured based on the percentage of correct predictions in the pairwise comparisons. A prediction is correct is the same headline (A or B) has a higher probability of being clicked than the other based on our probability prediction.

In [6]:
def generate_pairwise_dataset(df):
    pairs = []
    pair_headers = ["embedding_a", "embedding_b", "impressions_a", "impressions_b", "clicks_a", "clicks_b", "prob_a_gt_b"]
    samples = 5000
    for test_id, group in df.groupby("clickability_test_id"):
        rows = list(zip(group["embedding"], group["impressions"], group["clicks"]))
        total_pairs = len(rows) * (len(rows) - 1) // 2
        if total_pairs == 0:
            continue

        for (embedding_a, impressions_a, clicks_a), (embedding_b, impressions_b, clicks_b) in combinations(rows, 2):
            alpha_a, beta_a = clicks_a + 1, impressions_a - clicks_a + 1
            alpha_b, beta_b = clicks_b + 1, impressions_b - clicks_b + 1
            draws_a = np.random.beta(alpha_a, beta_a, size=samples)
            draws_b = np.random.beta(alpha_b, beta_b, size=samples)
            prob_a_gt_b = float((draws_a > draws_b).mean())
            pairs.append([embedding_a, embedding_b, impressions_a, impressions_b, clicks_a, clicks_b, prob_a_gt_b])
    return pd.DataFrame(pairs, columns=pair_headers)

In [7]:
train_pairwise_df = generate_pairwise_dataset(train_embeddings_df)
validation_pairwise_df = generate_pairwise_dataset(validation_embeddings_df)
test_pairwise_df = generate_pairwise_dataset(test_embeddings_df)

In [8]:
print(f"Number of rows in train_pairwise_df: {len(train_pairwise_df)}")
print(f"Number of rows in validation_pairwise_df: {len(validation_pairwise_df)}")
print(f"Number of rows in test_pairwise_df: {len(test_pairwise_df)}")

Number of rows in train_pairwise_df: 111005
Number of rows in validation_pairwise_df: 23324
Number of rows in test_pairwise_df: 24324


Then, we want to obtain the difference or distance between the embeddings of the two headlines. It's what we will use to train our model.

In [9]:
def generate_embeddings_difference(df):
    df = df[["embedding_a", "embedding_b", "impressions_a", "impressions_b", "clicks_a", "clicks_b", "prob_a_gt_b"]].copy()
    df["embedding_difference"] = df["embedding_a"] - df["embedding_b"]
    return df

In [10]:
train_diff_df = generate_embeddings_difference(train_pairwise_df)
validation_diff_df = generate_embeddings_difference(validation_pairwise_df)
test_diff_df = generate_embeddings_difference(test_pairwise_df)

## Linear Regression

Once we have the embeddings difference for each pairwise comparison, the label will be the probability of the headline A being clicked more than headline B. This is defined by "prob_a_gt_b" and has been generated by sampling from the beta distribution.

We don't use Logistic Regression because we don't have 0s and 1s, we have the probabilities. If we transform them to 0s and 1s, we will lose information.

I'll use Linear Regression and I'll apply a logit transformation (Inverse of the sigmoid function) to the label.

In [11]:
EPS = 1e-6
def train_linear_regression(df):
    
    y = df["prob_a_gt_b"].to_numpy()
    y_clipped = np.clip(y, EPS, 1 - EPS)
    target = logit(y_clipped)
    X = np.vstack(df["embedding_difference"].to_numpy())

    base_model = LinearRegression()
    base_model.fit(X, target)

    print("Trained linear regression")
    print(f"Coefficients shape: {base_model.coef_.shape}, intercept: {base_model.intercept_}")

    return base_model

In [13]:
lr_model = train_linear_regression(train_diff_df)

Trained linear regression
Coefficients shape: (384,), intercept: 0.3385179936885834


In [14]:
def evaluate_linear_regression(base_model, df):
    y = df["prob_a_gt_b"].to_numpy()
    y_clipped = np.clip(y, EPS, 1 - EPS)
    target = logit(y_clipped)
    X = np.vstack(df["embedding_difference"].to_numpy())
    pred_logit = base_model.predict(X)
    pred_prob = expit(pred_logit)
    true_label = y > 0.5
    pred_label = pred_prob > 0.5
    accuracy = (true_label == pred_label).mean()
    return accuracy

In [16]:
accuracy = evaluate_linear_regression(lr_model, validation_diff_df)
print(f"Validation accuracy: {accuracy}")

# Evaluate on test set
accuracy = evaluate_linear_regression(lr_model, test_diff_df)
print(f"Test accuracy: {accuracy}")

Validation accuracy: 0.6050420168067226
Test accuracy: 0.6081647755303404


The simple Linear Regression, with a minimal embedding model, gave us an accuracy of 60%. Already greater than what we were able to obtain with traditional NLP.

## Neural Network

Now, we will try to use a Neural Network to see if we can obtain better results.

The neural network will be a simple feedforward network with two layers.

In [17]:
# Create tensors
def build_torch_dataset(df):
    X = np.vstack(df["embedding_difference"].to_numpy()).astype(np.float32)
    y = df["prob_a_gt_b"].to_numpy().astype(np.float32)
    X_tensor = torch.from_numpy(X)
    y_tensor = torch.from_numpy(y).unsqueeze(1)
    return TensorDataset(X_tensor, y_tensor)

In [23]:
def train_torch_model(train_df, validation_df, epochs=3, batch_size=256, lr=1e-3):
    train_ds = build_torch_dataset(train_df)
    validation_ds = build_torch_dataset(validation_df)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    validation_loader = DataLoader(validation_ds, batch_size=batch_size, shuffle=False)

    input_dim = train_ds.tensors[0].shape[1]
    model = nn.Sequential(
        nn.Linear(input_dim, 128),
        nn.ReLU(),
        nn.Linear(128, 1),
    )
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
        avg_loss = total_loss / len(train_ds)
        print(f"Epoch {epoch+1}/{epochs} - train loss: {avg_loss:.4f}")

        # Validation loss
        model.eval()
        with torch.no_grad():
            val_loss = 0.0
            for xb, yb in validation_loader:
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
            val_loss /= len(validation_ds)
        print(f"Epoch {epoch+1}/{epochs} - val loss: {val_loss:.4f}")
        model.train()
        
    return model

In [24]:
def evaluate_torch_model(model, df):
    ds = build_torch_dataset(df)
    loader = DataLoader(ds, batch_size=512, shuffle=False)
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            logits = model(xb)
            prob = torch.sigmoid(logits)
            preds.append(prob.squeeze(1).cpu().numpy())
    pred_prob = np.concatenate(preds)
    true_label = ds.tensors[1].squeeze(1).numpy() > 0.5
    pred_label = pred_prob > 0.5
    accuracy = (true_label == pred_label).mean()
    return accuracy

In [30]:
# Train and evaluate PyTorch model
torch_model = train_torch_model(train_diff_df, validation_diff_df, epochs=10, batch_size=256, lr=1e-4)
val_acc = evaluate_torch_model(torch_model, validation_diff_df)
test_acc = evaluate_torch_model(torch_model, test_diff_df)
print(f"Validation accuracy: {val_acc}")
print(f"Test accuracy: {test_acc}")

Epoch 1/10 - train loss: 0.6840
Epoch 1/10 - val loss: 0.6744
Epoch 2/10 - train loss: 0.6671
Epoch 2/10 - val loss: 0.6651
Epoch 3/10 - train loss: 0.6619
Epoch 3/10 - val loss: 0.6635
Epoch 4/10 - train loss: 0.6601
Epoch 4/10 - val loss: 0.6632
Epoch 5/10 - train loss: 0.6592
Epoch 5/10 - val loss: 0.6629
Epoch 6/10 - train loss: 0.6585
Epoch 6/10 - val loss: 0.6629
Epoch 7/10 - train loss: 0.6580
Epoch 7/10 - val loss: 0.6631
Epoch 8/10 - train loss: 0.6576
Epoch 8/10 - val loss: 0.6630
Epoch 9/10 - train loss: 0.6572
Epoch 9/10 - val loss: 0.6630
Epoch 10/10 - train loss: 0.6569
Epoch 10/10 - val loss: 0.6630
Validation accuracy: 0.607614474361173
Test accuracy: 0.6119470481828646


The Neural Network gave us an accuracy of 61%. We'll try with a more complex embedding model.

## More complex embedding model

We'll use the BAAI/bge-base-en-v1.5 model to generate embeddings for the headlines.

In [31]:
encoder = SentenceTransformer("BAAI/bge-base-en-v1.5")
train_embeddings_df = generate_embeddings_dataset(train_df)
validation_embeddings_df = generate_embeddings_dataset(validation_df)
test_embeddings_df = generate_embeddings_dataset(test_df)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [32]:
train_pairwise_df = generate_pairwise_dataset(train_embeddings_df)
validation_pairwise_df = generate_pairwise_dataset(validation_embeddings_df)
test_pairwise_df = generate_pairwise_dataset(test_embeddings_df)

In [33]:
train_diff_df = generate_embeddings_difference(train_pairwise_df)
validation_diff_df = generate_embeddings_difference(validation_pairwise_df)
test_diff_df = generate_embeddings_difference(test_pairwise_df)

In [34]:
lr_model = train_linear_regression(train_diff_df)
accuracy = evaluate_linear_regression(lr_model, validation_diff_df)
print(f"Validation accuracy: {accuracy}")

# Evaluate on test set
accuracy = evaluate_linear_regression(lr_model, test_diff_df)
print(f"Test accuracy: {accuracy}")

Trained linear regression
Coefficients shape: (768,), intercept: 0.30960404872894287
Validation accuracy: 0.6268221574344023
Test accuracy: 0.6352984706462753


In [37]:
# Train and evaluate PyTorch model
torch_model = train_torch_model(train_diff_df, validation_diff_df, epochs=20, batch_size=256, lr=1e-4)
val_acc = evaluate_torch_model(torch_model, validation_diff_df)
test_acc = evaluate_torch_model(torch_model, test_diff_df)
print(f"Validation accuracy: {val_acc}")
print(f"Test accuracy: {test_acc}")

Epoch 1/50 - train loss: 0.6800
Epoch 1/50 - val loss: 0.6664
Epoch 2/50 - train loss: 0.6583
Epoch 2/50 - val loss: 0.6571
Epoch 3/50 - train loss: 0.6527
Epoch 3/50 - val loss: 0.6550
Epoch 4/50 - train loss: 0.6505
Epoch 4/50 - val loss: 0.6543
Epoch 5/50 - train loss: 0.6493
Epoch 5/50 - val loss: 0.6537
Epoch 6/50 - train loss: 0.6484
Epoch 6/50 - val loss: 0.6535
Epoch 7/50 - train loss: 0.6477
Epoch 7/50 - val loss: 0.6536
Epoch 8/50 - train loss: 0.6471
Epoch 8/50 - val loss: 0.6534
Epoch 9/50 - train loss: 0.6466
Epoch 9/50 - val loss: 0.6534
Epoch 10/50 - train loss: 0.6462
Epoch 10/50 - val loss: 0.6533
Epoch 11/50 - train loss: 0.6458
Epoch 11/50 - val loss: 0.6532
Epoch 12/50 - train loss: 0.6455
Epoch 12/50 - val loss: 0.6532
Epoch 13/50 - train loss: 0.6451
Epoch 13/50 - val loss: 0.6532
Epoch 14/50 - train loss: 0.6448
Epoch 14/50 - val loss: 0.6531
Epoch 15/50 - train loss: 0.6445
Epoch 15/50 - val loss: 0.6532
Epoch 16/50 - train loss: 0.6442
Epoch 16/50 - val loss: 0

Using a slightly more complex embedding model the accuracy went up to 63%.